# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)
qm9_data = mol_instruction_dataset['property_prediction']
qm9_homo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo
            )
qm9_lumo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_lumo
            )
qm9_homo_lumo_gap_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo_lumo_gap
            )

In [3]:
def get_qm9_data_list(
        qm9_data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(qm9_data)))
    for i in iter_bar:
        data_instance = qm9_data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [4]:
list_qm9_homo_tr_data, list_qm9_homo_te_data = get_qm9_data_list(
    qm9_data=qm9_homo_data,
    instruction_templates=instructions_smol.qm9_homo,
    task="qm9_homo",
)

homo_dict = {
    'task_name': 'qm9_homo',
    "train": list_qm9_homo_tr_data,
    "test": list_qm9_homo_te_data
}

list_qm9_lumo_tr_data, list_qm9_lumo_te_data = get_qm9_data_list(
    qm9_data=qm9_lumo_data,
    instruction_templates=instructions_smol.qm9_lumo,
    task="qm9_lumo",
)

lumo_dict = {
    'task_name': 'qm9_lumo',
    "train": list_qm9_lumo_tr_data,
    "test": list_qm9_lumo_te_data
}

list_qm9_homo_lumo_gap_tr_data, list_qm9_homo_lumo_gap_te_data = get_qm9_data_list(
    qm9_data=qm9_homo_lumo_gap_data,
    instruction_templates=instructions_smol.qm9_homo_lumo_gap,
    task="qm9_homo_lumo_gap",
)

gap_dict = {
    'task_name': 'qm9_homo_lumo_gap',
    "train": list_qm9_homo_lumo_gap_tr_data,
    "test": list_qm9_homo_lumo_gap_te_data
}



for data_dict in [
    homo_dict,
    lumo_dict,
    gap_dict
]:
    for split in ["train", "test"]:
        list_data = data_dict[split]
        task_name = data_dict['task_name']
        
        dataset = datasets.Dataset.from_list(list_data)
        dataset.save_to_disk(
            f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_{task_name}_0228"
        )

100%|██████████| 684/684 [00:00<00:00, 1992.92it/s]


120062 684


100%|██████████| 642/642 [00:00<00:00, 1980.33it/s]


120111 642


100%|██████████| 661/661 [00:00<00:00, 1993.39it/s]


119940 661


Saving the dataset (1/1 shards): 100%|██████████| 661/661 [00:00<00:00, 69762.59 examples/s]


In [5]:
list_qm9_train = homo_dict["train"] + lumo_dict["train"] + gap_dict["train"]
list_qm9_test = homo_dict["test"] + lumo_dict["test"] + gap_dict["test"]

In [6]:
len(list_qm9_train), len(list_qm9_test)

(360113, 1987)

In [7]:
qm9_trainset = datasets.Dataset.from_list(list_qm9_train)
qm9_trainset.save_to_disk(
    f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_0228"
)

qm9_testset = datasets.Dataset.from_list(list_qm9_test)
qm9_testset.save_to_disk(
    f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_0228"
)
qm9_testset.save_to_disk(
    f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_validation_qm9_0228"
)

Saving the dataset (1/1 shards): 100%|██████████| 1987/1987 [00:00<00:00, 148148.29 examples/s]


In [5]:
gap_trainset = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_lumo_gap_0224')
gap_testset = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_lumo_gap_0224')

In [11]:
gap_trainset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 119940
})

In [12]:
gap_testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 661
})

In [3]:
trainset = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train')
testset = datasets.load_from_disk('/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test')

In [6]:
trainset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 4959023
})

In [7]:
testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 32851
})

In [8]:
gap_filtered_testset = testset.filter(lambda x:"gap" not in x["task"])
previous_gap_testset = testset.filter(lambda x:"gap" in x["task"])

Filter: 100%|██████████| 32851/32851 [00:17<00:00, 1895.99 examples/s]


In [17]:
previous_gap_testset[0]

{'task': 'qm9_homo_lumo_gap',
 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [5, 0, 3, 5, 0, 0, 1, 1, 1],
  [6, 0, 2, 5, 0, 0, 1, 1, 1],
  [7, 0, 2, 5, 0, 0, 1, 1, 1],
  [6, 0, 3, 5, 2, 0, 1, 0, 0],
  [5, 0, 4, 5, 2, 0, 2, 0, 0],
  [7, 0, 2, 5, 1, 0, 2, 0, 0]],
 'edge_index': [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 3, 6, 2, 7, 7, 8, 5, 1],
  [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 3, 7, 2, 8, 7, 1, 5]],
 'edge_attr': [[0, 0, 0],
  [0, 0, 0],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [3, 0, 1],
  [0, 0, 1],
  [0, 0, 1],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [3, 0, 1],
  [3, 0, 1]],
 'additional_x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0]],
 'additional_edge_index': [[0, 1], [1, 0]],
 'additional_edge_attr': [[0, 0, 0], [0, 0, 0]],
 'prompt_text': '<s>[INST] You are a helpful assistant for molecular chemistry, to address tasks including molecular property classifica

In [8]:
대학원생 = [
    '찬희', 
    '한범',
    '동환',
    '효민',
    '현지',
    '용준',
    '윤진',
    '유헌',
    '지연',
    '송성',
    '연규',
    '용진',
    '서환',
    '지영',
]

학부생 = [
    '수민',
    '윤경',
    '유진',
    '지우',
    '하연'
]

In [9]:
len(presenters)

19

In [18]:
gap_testset[0]

{'task': 'qm9_homo_lumo_gap',
 'x': [[7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 0, 1],
  [7, 0, 2, 5, 0, 0, 1, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1]],
 'edge_index': [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 6, 1, 7, 4],
  [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 6, 1, 6, 4, 7]],
 'edge_attr': [[1, 0, 1],
  [1, 0, 1],
  [0, 0, 1],
  [0, 0, 1],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0]],
 'additional_x': [[7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 0, 0, 1, 0, 1],
  [7, 0, 2, 5, 0, 0, 1, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1]],
 'additional_edge_index': [[0,
   1,
   1,
   2,
   2,
   3,
   3,
   4,

In [13]:
gap_filtered_testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 32167
})

In [14]:
previous_gap_testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 684
})

In [ ]:
gap_fixed_testdata = [
    gap_filtered_testset,
    gap_testset
]

# concatenate datasets
gap_fixed_testset = datasets.concatenate_datasets(gap_fixed_testdata)
gap_fixed_testset.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_gap_fixed_0224'
)

Saving the dataset (1/1 shards): 100%|██████████| 32828/32828 [00:02<00:00, 13985.02 examples/s]


In [24]:
gap_testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'input_mol_string', 'prompt_text', 'target_text'],
    num_rows: 661
})

In [ ]:
gap_fixed_testset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 32828
})

In [29]:
previous_gap_trainset = trainset.filter(
        lambda x:"gap" in x["task"],
        num_proc=200
    )

Filter (num_proc=200): 100%|██████████| 4959023/4959023 [00:25<00:00, 194550.82 examples/s]


In [30]:
previous_gap_trainset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 117660
})

In [31]:
previous_gap_trainset[0]

{'task': 'qm9_homo_lumo_gap',
 'x': [[6, 0, 1, 5, 0, 0, 0, 0, 0],
  [5, 0, 2, 5, 0, 0, 0, 0, 0],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [6, 0, 2, 5, 0, 0, 1, 0, 1],
  [5, 0, 3, 5, 0, 0, 1, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [5, 0, 4, 5, 2, 0, 2, 0, 1],
  [6, 0, 3, 5, 0, 0, 1, 0, 1]],
 'edge_index': [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 7, 8, 8, 2, 8, 5],
  [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 6, 8, 7, 2, 8, 5, 8]],
 'edge_attr': [[2, 0, 0],
  [2, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [1, 0, 1],
  [1, 0, 1],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 1],
  [0, 0, 1]],
 'additional_x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0]],
 'additional_edge_index': [[0, 1], [1, 0]],
 'additional_edge_attr': [[0, 0, 0], [0, 0, 0]],
 'prompt_text': '<s>[INST] You are a helpful assistant for molecular chemistry, to address tasks i

In [4]:
gap_filtered_trainset = trainset.filter(
        lambda x:"gap" not in x["task"],
        num_proc=200
    )

In [28]:
gap_filtered_trainset

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string'],
    num_rows: 4841363
})

In [33]:
gap_trainset[0]

{'task': 'qm9_homo_lumo_gap',
 'x': [[7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 1, 0, 1, 0, 0],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [6, 0, 3, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 3, 5, 1, 0, 1, 0, 0],
  [7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 1, 0, 1, 0, 1],
  [5, 0, 3, 5, 1, 0, 1, 0, 1]],
 'edge_index': [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 4, 7, 7, 8, 8, 2],
  [1, 0, 2, 1, 3, 2, 4, 3, 5, 4, 6, 5, 7, 4, 8, 7, 2, 8]],
 'edge_attr': [[1, 0, 0],
  [1, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [1, 0, 0],
  [1, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [1, 0, 0],
  [1, 0, 0],
  [0, 0, 0],
  [0, 0, 0]],
 'additional_x': [[7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 1, 0, 1, 0, 0],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [6, 0, 3, 5, 1, 0, 2, 0, 1],
  [5, 0, 4, 5, 1, 0, 2, 0, 1],
  [5, 0, 3, 5, 1, 0, 1, 0, 0],
  [7, 0, 1, 5, 0, 0, 1, 0, 0],
  [5, 0, 3, 5, 1, 0, 1, 0, 1],
  [5, 0, 3, 5, 1, 0, 1, 0, 1]],
 'addition

In [6]:
gap_fixed_traindata = [
    gap_filtered_trainset,
    gap_trainset
]

# concatenate datasets
gap_fixed_trainset = datasets.concatenate_datasets(gap_fixed_traindata)
gap_fixed_trainset.save_to_disk(
    '/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_gap_fixed_0224'
)

Saving the dataset (50/50 shards): 100%|██████████| 4961303/4961303 [04:37<00:00, 17876.63 examples/s]
